# Weight demo

This notebook shows how to connect and drive a kern weight scale from Python.

## 1. Initialisation

In [ ]:
import time
import csv
import threading
import numpy as np


from enderscope import WeightScale
from enderscope import Stage, ScanPatterns
from enderscope import Magnetometer, MagnetometerLogger
from pathlib import Path
from datetime import datetime

In [5]:
scale = WeightScale('COM11', baud_rate=9600)   # balance
ender = Stage('COM12', 115200)                   # plateau déplacement (X, Y, Z)
mag = Magnetometer('COM3', baud_rate=115200)    # magnétomètre (5 x MLX90393)

print(scale)
print(ender)
print(mag)

## 2. Positionnement / Force

### Homing et positionnement de la balance

<div style="background-color:#ffe0e0; border: 3px solid #d90000; border-radius: 8px; padding: 14px;">
<h2 style="color:#d90000; margin-top:0;">⚠️ ATTENTION ⚠️</h2>
<p style="font-size:16px; color:#7a0000;"><strong>RETIRER L'INDENTEUR avant de lancer le homing ci-dessous !</strong><br></p>
</div>

In [ ]:
ender.home()
ender.set_speed(6000)
ender.move_relative(0, 0, 60)
ender.finish_moves()

### Point de départ

Coordonnées du premier point de mesure à définit selon la manip (X, Y, Z)

In [14]:
x_depart = 100
y_depart = 150
z_depart = 48.5

ender.move_absolute(x_depart, y_depart, z_depart)
ender.finish_moves()

### Tare de la balance

In [17]:
scale.perform_tare()

Empty answer from scale
Empty answer from scale
Empty answer from scale


### Fonction d'indentation

In [ ]:
import time
def apply_force(max_force=100, speed=60):
    force = 0
    ender.set_speed(speed)
    while force < max_force:
        if(force < 0.8 * max_force):
            ender.move_relative(0,0,-0.1)
        else:
            ender.set_speed(int(speed/2))
            ender.move_relative(0,0,-0.05)
        ender.finish_moves()
        force = scale.get_instant_reading()
    time.sleep(0.5)
    force = scale.get_instant_reading()
    end_pos = ender.get_position()
    #print (f"position: {end_pos} - force: {force}")
    return force, end_pos

## 3. Magnétomètre

### Lecture brute

In [24]:
values = mag.get_reading()
print(values)

[4120.80029296875, -230.40000915527344, 317.5039978027344, 3650.400146484375, 604.800048828125, 1560.416015625, -784.800048828125, -3184.800048828125, -1467.488037109375, 28.80000114440918, 2714.400146484375, 34.847999572753906, 165.60000610351562, 1264.800048828125, -7.74399995803833]


### Baseline

Moyenne de quelques lectures au repos pour corriger les mesures.

In [27]:
baseline = mag.acquire_baseline(num_samples=5)
print('Baseline magnétomètre :', baseline)

Baseline magnétomètre : [ 4125.60009766  -225.6000061    321.37600708  3648.96020508
   603.36002197  1556.54394531  -786.24001465 -3182.88012695
 -1468.26242676    31.20000076  2712.            30.97599983
   167.04000244  1268.64008789    -6.19519997]


## 4. Acquisition & enregistrement CSV

### Dossier de sortie

Chaque prise de mesure crée son propre dossier, nommé par la date/heure et le type de mesure
(ex: `20260918_143210_mesure_combinee`). 

In [31]:
def create_output_dir(measurement_type: str) -> Path:
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = Path(f'{timestamp_str}_{measurement_type}')
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir

### Plage de force

In [34]:
force_targets = np.arange(100, 700, 200)  
print(force_targets)

[100 300 500]


### Motif de scan

Grille de positions autour du point de départ.

In [37]:
origin = [x_depart, y_depart] # coordinates of first position
steps = [10,-10] # x and y shifts to next item
positions = ScanPatterns.snake(3,3) * steps + origin
#ScanPatterns.plot_path(positions, field=(0,0),title='snake scan')

### a) Boucle de scan — mesure combinée (position + force)

1. création du dossier de sortie horodaté pour cette mesure
2. démarrage du `MagnetometerLogger` (flux indépendant, `sensor.csv` / `sensor_post_baselines.csv`)
3. boucle de scan : indentation à chaque position, écriture de `states.csv` avec son propre timestamp
4. fin du scan

In [ ]:
output_dir = create_output_dir('mesure_combinee')
print('Dossier de sortie :', output_dir)

mag_logger = MagnetometerLogger(
    mag, baseline,
    sensor_path=output_dir / 'sensor.csv',
    corrected_path=output_dir / 'sensor_post_baselines.csv',
)
mag_logger.start()

try:
    with open(output_dir / 'states.csv', 'w', newline='') as f_states:
        writer_states = csv.writer(f_states)

        for scan_pos in positions:
            ender.move_position(scan_pos)
            ender.finish_moves()
            start_pos = ender.get_position()

            scale.perform_tare()  # remise à zéro avant d'indenter à cette position

            for target_force in force_targets:
                force, end_pos = apply_force(target_force)
                print(f'z: {end_pos[2]} - force: {force}g (cible: {target_force}g)')

                timestamp = time.time()
                x, y, z = end_pos
                writer_states.writerow([timestamp, x, y, z, force])

            ender.set_speed(6000)
            ender.move_position(start_pos)
            ender.finish_moves()
finally:
    mag_logger.stop()
    mag_logger.join()

Dossier de sortie : 20260918_191428_mesure_combinee
Empty answer from scale
z: 45.85 - force: 101.8g (cible: 100g)
z: 42.7 - force: 304.5g (cible: 300g)
z: 40.45 - force: 505.5g (cible: 500g)
Empty answer from scale
z: 45.5 - force: 101.4g (cible: 100g)
z: 42.35 - force: 302.1g (cible: 300g)
z: 39.95 - force: 506.5g (cible: 500g)
Empty answer from scale
Empty answer from scale
z: 45.05 - force: 100.9g (cible: 100g)
z: 41.9 - force: 306.0g (cible: 300g)
z: 39.7 - force: 501.4g (cible: 500g)
Empty answer from scale
z: 45.2 - force: 102.2g (cible: 100g)
z: 42.05 - force: 303.8g (cible: 300g)
z: 39.65 - force: 506.4g (cible: 500g)
Empty answer from scale
z: 45.7 - force: 101.8g (cible: 100g)
z: 42.65 - force: 301.6g (cible: 300g)
z: 40.3 - force: 505.2g (cible: 500g)
Empty answer from scale
z: 45.9 - force: 104.5g (cible: 100g)
z: 42.8 - force: 303.3g (cible: 300g)
z: 40.45 - force: 502.7g (cible: 500g)
Empty answer from scale
z: 46.05 - force: 100.8g (cible: 100g)
z: 42.9 - force: 307.4g 

### b) Boucle de scan — mesure position seule
TODO

### c) Boucle de scan — mesure force seule
TODO